# M0.5 Dashboard — read JSONL training/gate logs and plot results

Reads `train.jsonl` and `gates.jsonl` from one or more `run_*` directories under `runs/m05/` and renders:
- loss curves (per component)
- branch mass over time (with collapse-zone shaded)
- per-module grad norms
- gate status table

Pass multiple run dirs in `RUN_DIRS` to overlay runs.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

RUN_DIRS = [
    # Edit this to point at your runs:
    # Path('runs/m05/phase_a_20260529_160903'),
]
if not RUN_DIRS:
    # Auto-discover under default location
    base = Path('runs/m05')
    if not base.exists():
        base = Path('/tmp/m05')
    RUN_DIRS = sorted([p for p in base.glob('*') if p.is_dir() and (p / 'train.jsonl').exists()])
RUN_DIRS

In [ ]:
def load_jsonl(p: Path) -> pd.DataFrame:
    if not p.exists():
        return pd.DataFrame()
    return pd.DataFrame([json.loads(line) for line in p.read_text().splitlines() if line.strip()])

runs = {p.name: {'train': load_jsonl(p / 'train.jsonl'), 'gates': load_jsonl(p / 'gates.jsonl')} for p in RUN_DIRS}
for name, r in runs.items():
    print(f'{name:>40s}  train_rows={len(r["train"])}  gate_rows={len(r["gates"])}')

## Loss curves

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 4))
for name, r in runs.items():
    df = r['train']
    if 'lm_loss' in df.columns:
        ax.plot(df['step'], df['lm_loss'], label=f'{name}', alpha=0.7)
ax.set_xlabel('step'); ax.set_ylabel('lm_loss'); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
plt.title('LM Loss')
plt.tight_layout(); plt.show()

## Branch masses — collapse detector

Shaded zone < 0.05 is the collapse threshold (G2 fails if min branch mass spends >= 50 of first 1K steps there).

In [ ]:
fig, axes = plt.subplots(len(runs) or 1, 1, figsize=(10, 3 * max(1, len(runs))), sharex=True, squeeze=False)
for i, (name, r) in enumerate(runs.items()):
    df = r['train']
    ax = axes[i, 0]
    for col, color in [('sliding_mass', 'C0'), ('selected_mass', 'C1'), ('compressed_mass', 'C2')]:
        if col in df.columns:
            ax.plot(df['step'], df[col], label=col, color=color, alpha=0.8)
    ax.axhspan(0, 0.05, color='red', alpha=0.1)
    ax.set_ylabel('branch_mass'); ax.set_title(name); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
axes[-1, 0].set_xlabel('step')
plt.tight_layout(); plt.show()

## Per-module grad norms

In [ ]:
fig, axes = plt.subplots(len(runs) or 1, 1, figsize=(10, 3 * max(1, len(runs))), sharex=True, squeeze=False)
for i, (name, r) in enumerate(runs.items()):
    df = r['train']
    ax = axes[i, 0]
    grad_cols = [c for c in df.columns if c.startswith('grad_norm_') and c != 'grad_norm_total']
    if 'grad_norm_total' in df.columns:
        ax.plot(df['step'], df['grad_norm_total'], label='total', lw=2, color='k')
    for c in grad_cols:
        ax.plot(df['step'], df[c], label=c.replace('grad_norm_', ''), alpha=0.6)
    ax.set_yscale('log'); ax.set_ylabel('grad_norm'); ax.set_title(name)
    ax.legend(fontsize=6, ncol=3); ax.grid(True, alpha=0.3)
axes[-1, 0].set_xlabel('step')
plt.tight_layout(); plt.show()

## Gate status table

In [ ]:
rows = []
for name, r in runs.items():
    for _, g in r['gates'].iterrows():
        rows.append({'run': name, **g.to_dict()})
if rows:
    g = pd.DataFrame(rows)[['run', 'gate_id', 'metric', 'value', 'threshold', 'status']]
    def hilite(s):
        m = {'pass': 'background-color:#cfe8cf;', 'fail':'background-color:#f5c6c6;',
             'stretch':'background-color:#fff4c2;', 'deferred':'background-color:#e0e0e0;'}
        return [m.get(v, '') for v in s]
    g.style.apply(hilite, subset=['status'])
else:
    print('no gates recorded')